# PathSnap — Member A: Exploratory Data Analysis & Feature Engineering

**Course:** PES University — UE24CS352A Machine Learning  
**Project:** PathSnap (Distributed Map-Matching & Route Reconstruction)  
**Component:** Member A — Data Processing, Candidate Generation, Label Alignment & Feature Engineering  

---
### Notebook Objectives:
1. Inspect dataset structure across 80 training and 10 student-test trips.
2. Demonstrate metric UTM coordinate projection per trip.
3. Perform high-performance candidate road segment generation using Shapely STRtree.
4. Perform sequence-monotonic ground-truth label alignment on training data.
5. Extract 24 geometric, directional, kinematic, and topological features.
6. Visualize trajectory patterns, feature correlations, and interactive Folium maps.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from member_a.config.settings import DEFAULT_DATA_ROOT, RESULTS_DIR, PLOTS_DIR, MAPS_DIR
from member_a.src.data.loader import load_trip
from member_a.src.projection.projection import project_trip, get_utm_epsg_for_coords
from member_a.src.candidates.generator import generate_candidates
from member_a.src.labels.alignment import align_route_labels
from member_a.src.features.engineering import build_features
from member_a.src.features.schema import split_features_and_identifiers, FEATURE_COLUMNS, TARGET_COLUMN
from member_a.src.viz.folium_maps import create_interactive_map

print('Environment initialized successfully.')

## 1. Dataset Inspection & Summary Across Partitions

In [ ]:
summary_df = pd.read_csv(RESULTS_DIR / 'dataset_summary.csv')
print(f'Total trips inspected: {len(summary_df)}')
summary_df.groupby('partition')[['num_gps', 'num_nodes', 'num_arcs', 'pct_1hz']].mean()

## 2. Load and Project a Sample Trip into Metric Space

In [ ]:
trip = load_trip(partition='train', trip_id='00000001', include_route=True)
print(f'Trip {trip.trip_id}: {trip.num_gps} GPS observations, {trip.num_nodes} nodes, {trip.num_arcs} arcs')
proj_track, proj_nodes, epsg = project_trip(trip)
print(f'Dynamically assigned UTM EPSG: {epsg}')
proj_track[['lon', 'lat', 'timestamp', 'x_m', 'y_m']].head()

## 3. Candidate Road Segment Generation (Spatial STRtree Index)

In [ ]:
candidates = generate_candidates(trip, search_radius_m=50.0, max_candidates=10)
print(f'Generated {len(candidates)} candidates ({len(candidates)/trip.num_gps:.1f} per GPS point)')
candidates.head(10)

## 4. Sequence-Monotonic Training Route Alignment & Labels

In [ ]:
labeled_candidates = align_route_labels(trip, candidates)
positives = labeled_candidates[labeled_candidates['label'] == 1]
print(f'Positive candidate matches: {len(positives)} / {trip.num_gps} ({len(positives)/trip.num_gps*100:.2f}% recall)')
labeled_candidates.head(10)

## 5. Candidate Feature Extraction & Schema Verification

In [ ]:
features_df = build_features(trip, labeled_candidates)
X, meta, y = split_features_and_identifiers(features_df)
print(f'ML Feature Matrix X shape: {X.shape}')
print(f'Metadata shape: {meta.shape}')
print(f'Target vector y shape: {y.shape}, positive rate: {y.mean():.4f}')
X.describe().T[['mean', 'std', 'min', '50%', 'max']]

## 6. Visualizing Precomputed EDA Charts

In [ ]:
from IPython.display import Image, display
plot_files = sorted(list(PLOTS_DIR.glob('*.png')))
for p in plot_files:
    print(f'Displaying: {p.name}')
    display(Image(filename=str(p)))

## 7. Interactive Folium Geographical Map

In [ ]:
map_file = MAPS_DIR / f'trip_{trip.trip_id}_interactive_map.html'
if not map_file.exists():
    create_interactive_map(trip, candidates, labeled_candidates, max_points=120, out_html_path=map_file)
print(f'Interactive Folium map ready at: {map_file.resolve()}')